In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
catalog = dbutils.widgets.get("catalog_param")

In [ ]:
from pyspark.sql.functions import broadcast, col, count

In [ ]:
orders_df      = spark.table(f"{catalog}.silver.orders_cleaned")
customers_df   = spark.table(f"{catalog}.silver.customers_cleaned")
products_df    = spark.table(f"{catalog}.silver.products_cleaned")
order_items_df = spark.table(f"{catalog}.silver.order_items_cleaned")

In [ ]:
broadcast_join = orders_df \
    .join(broadcast(customers_df), "customer_id") \
    .join(order_items_df, "order_id") \
    .join(broadcast(products_df), "product_id")

print("✅ Broadcast join done")
broadcast_join.display()

In [ ]:
#orders_df.cache()
#customers_df.cache()


In [ ]:

orders_repartitioned = orders_df.repartition(8, "customer_id")
print("Repartition done - repartitioned by customer_id into 8 partitions")

orders_coalesced = orders_df.coalesce(2)
print("Coalesce done — reduced to 2 partitions")


orders_repartitioned.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.orders_repartitioned")
print(" Repartitioned table written successfully")

orders_coalesced.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.orders_coalesced")
print("Coalesced table written successfully")


In [ ]:

from pyspark.sql.functions import concat, lit, rand, floor


NUM_SALTS = 5

orders_salted = orders_df \
    .withColumn("salt", (floor(rand() * NUM_SALTS)).cast("int")) \
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt")))

customers_salted = customers_df \
    .withColumn("salt", col("customer_id") % NUM_SALTS) \
    .withColumn("salted_key", concat(col("customer_id"), lit("_"), col("salt")))

skew_join = orders_salted.join(customers_salted, "salted_key")

print("Skew handling done")

In [ ]:

# fact_sales is partitioned by order_date
# Querying with order_date filter = only reads relevant partitions
# Much faster than full table scan

pruned = spark.table(f"{catalog}.gold.fact_sales") \
    .filter(col("order_date") >= "2024-01-01")

print(f"Partition pruned row count: {pruned.count()}")
print("Partition pruning done")